# Basic MARL environment

In [1]:
!pip install gymnasium pettingzoo pygame stable-baselines3[extra] supersuit


INFO: pip is looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 8.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 7.0 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: pandas━━━━━━━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: pandas 2.1.3 0/2 [numpy]
    Uninstalling pandas-2.1.3:0m╺━━━━━━━━━━━━━━━━━━━ 1/2 [pandas]
      Successfully uninstalled pandas-2.1.3━━━━━━━━━━━━━━━━━━━ 1/2 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandapower 2.14.11 requir

In [2]:
import numpy as np
import functools

# Gym env
import gymnasium as gym
from gymnasium import spaces
from gymnasium.spaces import Discrete, MultiDiscrete
from gymnasium.utils import seeding

# MARL env using petting zoo (ParallelEnv: run all agents at simultaneously)
from pettingzoo.utils.env import ParallelEnv
import random
from pettingzoo.utils import parallel_to_aec, wrappers


In [ ]:
def env(render_mode=None):
    """
    The env function often wraps the environment in wrappers by default.
    You can find full documentation for these methods
    elsewhere in the developer documentation.
    """
    internal_render_mode = render_mode if render_mode != "ansi" else "human"
    env = raw_env(render_mode=internal_render_mode)
    # This wrapper is only for environments which print results to the terminal
    if render_mode == "ansi":
        env = wrappers.CaptureStdoutWrapper(env)
    # this wrapper helps error handling for discrete action spaces
    env = wrappers.AssertOutOfBoundsWrapper(env)
    # Provides a wide variety of helpful user errors
    # Strongly recommended
    env = wrappers.OrderEnforcingWrapper(env)
    return env

def raw_env(render_mode=None):
    """
    To support the AEC API, the raw_env() function just uses the from_parallel
    function to convert from a ParallelEnv to an AEC env
    """
    env = parallel_env(render_mode=render_mode)
    env = parallel_to_aec(env)
    return env

class parallel_env(ParallelEnv):
    metadata = {
        "name": "mouse_cat_cheese", 
        "render_modes": ["human"], 
        "render_fps": 4
    }

    def __init__(self, grid_size=5, vision_range=1, max_steps=50, render_mode=None):
        self.grid_size = grid_size
        self.vision_range = vision_range
        self.max_steps = max_steps

        self.possible_agents = ["mouse", "cat"]
        self.agents = self.possible_agents[:]

        self.action_spaces = {a: spaces.Discrete(4) for a in self.possible_agents}
        obs_shape = ((2 * vision_range + 1), (2 * vision_range + 1))
        self.observation_spaces = {
            a: spaces.Box(low=-1, high=2, shape=obs_shape, dtype=np.int8)
            for a in self.possible_agents
        }

        self.timestep = 0
        self.render_mode = render_mode

    def reset(self, seed=None, options=None):
        if seed is not None:
            self.np_random, self.np_random_seed = seeding.np_random(seed)
        
        self.agents = self.possible_agents[:]
        self.timestep = 0

        # mouse position
        self.mouse_pos = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)

        # cat position (ensure not same as mouse)
        while True:
            self.cat_pos = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)
            if not np.array_equal(self.cat_pos, self.mouse_pos):
                break

        # cheese position (ensure not same as mouse/cat)
        while True:
            self.cheese_pos = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)
            if (
                not np.array_equal(self.cheese_pos, self.mouse_pos)
                and not np.array_equal(self.cheese_pos, self.cat_pos)
            ):
                break

        observations = self._get_all_obs()

        infos = {agent: {} for agent in self.agents}
        return observations, infos

    def step(self, actions):
        if not actions:
            self.agents = []
            return {}, {}, {}, {}, {}
        
        rewards = {a: -0.1 for a in self.agents}
        terminations = {agent: False for agent in self.agents}

        # Move mouse
        self.mouse_pos = self._move(self.mouse_pos, actions["mouse"])

        # Move cat
        self.cat_pos = self._move(self.cat_pos, actions["cat"])

        # Check win/lose conditions
        if np.array_equal(self.mouse_pos, self.cheese_pos):
            rewards = {"mouse": 1, "cat": -1}
            terminations = {a: True for a in self.agents}

        elif np.array_equal(self.cat_pos, self.mouse_pos):
            rewards = {"mouse": -1, "cat": 1}
            terminations = {a: True for a in self.agents}

        # Max steps → draw
        truncations = {a: False for a in self.agents}
        if self.timestep > 100:
            rewards = {"mouse": 0, "cat": 0}
            truncations = {"mouse": True, "cat": True}
        self.timestep += 1
        terminations["__all__"] = all(terminations.values())

        obs = self._get_all_obs()

        infos = {agent: {} for agent in self.agents}
        
        if any(terminations.values()) or all(truncations.values()):
            self.agents = []

        if self.render_mode == "human":
            self.render()

        return obs, rewards, terminations, truncations, infos

    def _move(self, pos, action):
        x, y = pos
        temp = np.array([x, y])
        if action == 0 and y > 0:  # Up
            y -= 1
        elif action == 1 and x < self.grid_size - 1:  # Right
            x += 1
        elif action == 2 and y < self.grid_size - 1:  # Down
            y += 1
        elif action == 3 and x > 0:  # Left
            x -= 1

        return np.array([x, y])

    def _get_obs(self, agent):
        """Return partial grid view for the given agent."""
        full_grid = np.zeros((self.grid_size, self.grid_size), dtype=np.int8)
        # Encoding: 0=empty, 1=goal, 2=self, -1=unseen
        gx, gy = self.cheese_pos
        full_grid[gy][gx] = 1
        if agent == "mouse":
            ax, ay = self.mouse_pos
            ox, oy = self.cat_pos
        else:
            ax, ay = self.cat_pos
            ox, oy = self.mouse_pos
        full_grid[ay][ax] = 2  # self
        full_grid[oy][ox] = 3  # opponent

        vr = self.vision_range
        view = np.full((2 * vr + 1, 2 * vr + 1), -1, dtype=np.int8)
        for dx in range(-vr, vr + 1):
            for dy in range(-vr, vr + 1):
                gx_ = ax + dx
                gy_ = ay + dy
                if 0 <= gx_ < self.grid_size and 0 <= gy_ < self.grid_size:
                    view[dy + vr][dx + vr] = full_grid[gy_][gx_]
        return view

    def _get_all_obs(self):
        return {
            agent: self._get_obs(agent) for agent in self.agents
        }
    
    def render(self):
        if self.render_mode is None:
            gym.logger.warn(
                "You are calling render method without specifying any render mode."
            )
            return

        grid = np.full((self.grid_size, self.grid_size), ".", dtype='U6')
        gx, gy = self.cheese_pos
        grid[gy][gx] = "Cheese"
        rx, ry = self.mouse_pos
        grid[ry][rx] = "Mouse"
        cx, cy = self.cat_pos
        grid[cy][cx] = "Cat"
        print("\n".join("".join(f"{cell:^6}" for cell in row) for row in grid))
        print()

    def close(self):
        pass

        # Observation space should be defined here.
    # lru_cache allows observation and action spaces to be memoized, reducing clock cycles required to get each agent's space.
    # If your spaces change over time, remove this line (disable caching).
    @functools.lru_cache(maxsize=None)
    def observation_space(self, agent):
        # gymnasium spaces are defined and documented here: https://gymnasium.farama.org/api/spaces/
        return MultiDiscrete([self.grid_size * self.grid_size] * 3)

    # Action space should be defined here.
    # If your spaces change over time, remove this line (disable caching).
    @functools.lru_cache(maxsize=None)
    def action_space(self, agent):
        return Discrete(4)


In [6]:
from pettingzoo.test import parallel_api_test
env = env(render_mode='human')
parallel_api_test(env.parrallel_env(), num_cycles=1_000_000)

TypeError: 'OrderEnforcingWrapper' object is not callable

In [61]:
obs, _ = env.reset(50)
env._get_all_obs()
env.render()

  .     .     .     .     .   
  .     .     .     .     .   
  .     .     .     .   Cheese
  .     .     .   Mouse   .   
  .     .     .    Cat    .   



In [53]:
env = mousecatParallel(render_mode="human")
observations, infos = env.reset(seed=42)

while env.agents:
    # this is where you would insert your policy
    actions = {agent: env.action_space(agent).sample() for agent in env.agents}

    observations, rewards, terminations, truncations, infos = env.step(actions)
env.close()

  .     .     .     .     .   
  .     .     .    Cat    .   
  .     .     .     .     .   
Mouse   .     .     .     .   
  .     .   Cheese  .     .   

  .     .     .     .     .   
  .     .     .     .     .   
  .     .     .    Cat    .   
Mouse   .     .     .     .   
  .     .   Cheese  .     .   

  .     .     .     .     .   
  .     .     .     .     .   
Mouse   .     .     .    Cat  
  .     .     .     .     .   
  .     .   Cheese  .     .   

  .     .     .     .     .   
  .     .     .     .     .   
  .   Mouse   .     .    Cat  
  .     .     .     .     .   
  .     .   Cheese  .     .   

  .     .     .     .     .   
  .     .     .     .     .   
  .     .     .    Cat    .   
  .   Mouse   .     .     .   
  .     .   Cheese  .     .   

  .     .     .     .     .   
  .     .     .    Cat    .   
  .     .     .     .     .   
  .     .     .     .     .   
  .   Mouse Cheese  .     .   

  .     .     .     .     .   
  .     .     .     .     .   
  

/app/venvs/An/lib/python3.12/site-packages/pettingzoo/utils/env.py:370: UserWarning: Your environment should override the action_space function. Attempting to use the action_spaces dict attribute.
  warnings.warn(


In [25]:
from __future__ import annotations

import glob
import os
import time

import supersuit as ss
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy

In [26]:
env.metadata

{'name': 'mouse_cat_cheese', 'render_modes': ['human'], 'render_fps': 4}

In [ ]:
def train_supersuit(
    env_fn, steps: int = 10_000, seed: int | None = 0, **env_kwargs
):
    # Train a single model to play as each agent in a cooperative Parallel environment
    env = env_fn.parallel_env(**env_kwargs)

    env.reset(seed=seed)

    print(f"Starting training on {str(env.metadata['name'])}.")

    env = ss.pettingzoo_env_to_vec_env_v1(env)
    env = ss.concat_vec_envs_v1(env, 8, num_cpus=2, base_class="stable_baselines3")

    # Note: Waterworld's observation space is discrete (242,) so we use an MLP policy rather than CNN
    model = PPO(
        MlpPolicy,
        env,
        verbose=3,
        learning_rate=1e-3,
        batch_size=256,
    )

    model.learn(total_timesteps=steps)

    model.save(f"{env.unwrapped.metadata.get('name')}_{time.strftime('%Y%m%d-%H%M%S')}")

    print("Model has been saved.")

    print(f"Finished training on {str(env.unwrapped.metadata['name'])}.")

    env.close()

In [28]:
env_fn = env
env_kwargs = {}

# Train a model (takes ~3 minutes on GPU)
train_supersuit(env_fn, steps=196_608, seed=0, **env_kwargs)


Starting training on mouse_cat_cheese.


AttributeError: 'mousecatParallel' object has no attribute 'render_mode'

## More complex environment

In [93]:
class MultiAgentGridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self, grid_size: int = 5, num_agents: int = 2, num_goals: int = 2, seed: int = None):
        super().__init__()
        self.grid_size = grid_size
        self.num_agents = num_agents
        self.num_goals = num_goals
        self.seed = seed
        self.np_random, _ = seeding.np_random(seed)

        # Action and observation spaces
        self.action_space = spaces.MultiDiscrete([4] * self.num_agents)  # Each agent: 0=Up, 1=Right, 2=Down, 3=Left
        self.observation_space = spaces.Tuple([spaces.Box(low=0, high=grid_size - 1, shape=(2,), dtype=np.int32) for _ in range(num_agents)])

        # Environment state
        self.agent_positions = []
        self.goal_positions = []
        self.agent_goal_indices = [0] * self.num_agents  # Per-agent goal progress
        self.agent_visited_goals = [[] for _ in range(num_agents)]
        self.walls = []
        self.reset()

    def reset(self, *, seed=None, options=None):
        self.np_random, _ = seeding.np_random(seed or self.seed)

        # Place agents at unique positions
        self.agent_positions = []
        while len(self.agent_positions) < self.num_agents:
            pos = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)
            if not any(np.array_equal(pos, a) for a in self.agent_positions):
                self.agent_positions.append(pos)

        # Place goals at unique positions
        self.goal_positions = []
        while len(self.goal_positions) < self.num_goals:
            pos = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)
            if not any(np.array_equal(pos, g) for g in self.goal_positions) and all(not np.array_equal(pos, a) for a in self.agent_positions):
                self.goal_positions.append(pos)

        # Place walls
        self.walls = []
        wall_count = self.np_random.integers(1, self.grid_size * self.grid_size // 5)
        while len(self.walls) < wall_count:
            pos = tuple(self.np_random.integers(0, self.grid_size, size=(2,)))
            if pos not in self.walls and all(not np.array_equal(pos, g) for g in self.goal_positions) and all(not np.array_equal(pos, a) for a in self.agent_positions):
                self.walls.append(pos)

        # Reset goal tracking
        self.agent_goal_indices = [0] * self.num_agents
        self.agent_visited_goals = [[] for _ in range(self.num_agents)]

        obs = [pos.copy() for pos in self.agent_positions]
        return obs, {}

    def step(self, actions):
        rewards = [-0.1] * self.num_agents
        terminations = [False] * self.num_agents
        truncations = [False] * self.num_agents

        new_positions = []

        for i, action in enumerate(actions):
            x, y = self.agent_positions[i]
            new_x, new_y = x, y

            if action == 0 and y > 0:
                new_y -= 1  # Up
            elif action == 1 and x < self.grid_size - 1:
                new_x += 1  # Right
            elif action == 2 and y < self.grid_size - 1:
                new_y += 1  # Down
            elif action == 3 and x > 0:
                new_x -= 1  # Left

            new_pos = np.array([new_x, new_y])
            if tuple(new_pos) in self.walls:
                new_pos = self.agent_positions[i]  # Hit wall
                rewards[i] = -1.5
            new_positions.append(new_pos)

        self.agent_positions = new_positions

        # Check goals
        for i, pos in enumerate(self.agent_positions):
            goal_idx = self.agent_goal_indices[i]
            if goal_idx < self.num_goals:
                current_goal = self.goal_positions[goal_idx]
                if np.array_equal(pos, current_goal):
                    rewards[i] = 1
                    self.agent_visited_goals[i].append(pos.copy())
                    self.agent_goal_indices[i] += 1
                    if self.agent_goal_indices[i] >= self.num_goals:
                        terminations[i] = True
                elif any(np.array_equal(pos, g) for g in self.goal_positions[:goal_idx]):
                    rewards[i] = -0.5  # Already visited goal
                elif any(np.array_equal(pos, g) for g in self.goal_positions[goal_idx+1:]):
                    rewards[i] = -1  # Wrong order

        obs = [pos.copy() for pos in self.agent_positions]
        info = {}
        return obs, rewards, terminations, truncations, info

    def render(self):
        grid = np.full((self.grid_size, self.grid_size), ".", dtype="U3")

        # Place goals
        for i, goal in enumerate(self.goal_positions):
            gx, gy = goal
            grid[gy][gx] = f"G{i+1}"

        # Place walls
        for wall in self.walls:
            x, y = wall
            grid[y][x] = "###"

        # Place agents
        for i, pos in enumerate(self.agent_positions):
            x, y = pos
            grid[y][x] = f"A{i+1}"

        print("\n".join(" ".join(f"{cell:^4}" for cell in row) for row in grid))
        for i in range(self.num_agents):
            print(f"Agent {i+1}: Goal {self.agent_goal_indices[i] + 1}/{self.num_goals}, Visited: {len(self.agent_visited_goals[i])}")
        print()

    def close(self):
        pass


In [71]:
env = MultiAgentGridWorldEnv()
obs, _ = env.reset()
env.render()

 A1   .    .    A2   G2 
###   .    .    .    .  
###  ###   .    G1   .  
 .   ###   .    .    .  
 .    .    .    .    .  
Agent 1: Goal 1/2, Visited: 0
Agent 2: Goal 1/2, Visited: 0



In [ ]:
for step in range(10):
    actions = [random.randint(0, 3) for _ in range(env.num_agents)]
    obs, rewards, terminations, _, _ = env.step(actions)
    print(f"Step {step} - Actions: {actions}, Rewards: {rewards}, Done: {terminations}")
    env.render()
    if all(terminations):
        break


Step 0 - Actions: [1, 2], Rewards: [-0.1, -0.1], Done: [False, False]
 .    A1   .    .    G2 
###   .    .    A2   .  
###  ###   .    G1   .  
 .   ###   .    .    .  
 .    .    .    .    .  
Agent 1: Goal 1/2, Visited: 0
Agent 2: Goal 1/2, Visited: 0

Step 1 - Actions: [1, 0], Rewards: [-0.1, -0.1], Done: [False, False]
 .    .    A1   A2   G2 
###   .    .    .    .  
###  ###   .    G1   .  
 .   ###   .    .    .  
 .    .    .    .    .  
Agent 1: Goal 1/2, Visited: 0
Agent 2: Goal 1/2, Visited: 0

Step 2 - Actions: [3, 0], Rewards: [-0.1, -0.1], Done: [False, False]
 .    A1   .    A2   G2 
###   .    .    .    .  
###  ###   .    G1   .  
 .   ###   .    .    .  
 .    .    .    .    .  
Agent 1: Goal 1/2, Visited: 0
Agent 2: Goal 1/2, Visited: 0

Step 3 - Actions: [3, 3], Rewards: [-0.1, -0.1], Done: [False, False]
 A1   .    A2   .    G2 
###   .    .    .    .  
###  ###   .    G1   .  
 .   ###   .    .    .  
 .    .    .    .    .  
Agent 1: Goal 1/2, Visited: 0
Agent 2

In [88]:
from pettingzoo import ParallelEnv
from gymnasium import spaces
import numpy as np

class ParallelMultiAgentGridWorld(ParallelEnv):
    metadata = {"render_modes": ["human"], "name": "multi_agent_gridworld_v0"}

    def __init__(self, grid_size=5, num_agents=2, num_goals=2, render_mode=None, seed=None):
        self.env = MultiAgentGridWorldEnv(grid_size, num_agents, num_goals, seed)
        self._num_agents = num_agents
        self.agents = [f"agent_{i}" for i in range(num_agents)]
        self.possible_agents = self.agents[:]
        self.observation_spaces = {
            agent: self.env.observation_space.spaces[i]
            for i, agent in enumerate(self.agents)
        }
        self.action_spaces = {
            agent: spaces.Discrete(4)  # 4 actions: up, right, down, left
            for agent in self.agents
        }
        self.render_mode = render_mode

    def reset(self, seed=None, options=None):
        observations, _ = self.env.reset(seed=seed)
        self.agents = self.possible_agents[:]
        return {agent: obs for agent, obs in zip(self.agents, observations)}

    def step(self, actions):
        action_list = [actions[agent] for agent in self.agents]
        observations, rewards, terminations, truncations, infos = self.env.step(action_list)

        obs_dict = {agent: obs for agent, obs in zip(self.agents, observations)}
        rew_dict = {agent: rew for agent, rew in zip(self.agents, rewards)}
        term_dict = {agent: term for agent, term in zip(self.agents, terminations)}
        trunc_dict = {agent: trunc for agent, trunc in zip(self.agents, truncations)}
        info_dict = {agent: {} for agent in self.agents}

        self.agents = [agent for agent in self.agents if not term_dict[agent] and not trunc_dict[agent]]

        return obs_dict, rew_dict, term_dict, trunc_dict, info_dict

    def render(self):
        self.env.render()

    def close(self):
        self.env.close()


In [91]:
temp = ParallelMultiAgentGridWorld()
temp.render()

 A2   G1   .    .    .  
 .    .    .    G2  ### 
###   .    .    .    .  
 .    .    A1   .    .  
 .    .    .    .   ### 
Agent 1: Goal 1/2, Visited: 0
Agent 2: Goal 1/2, Visited: 0



In [20]:
temp.agents

['agent_0', 'agent_1']

In [25]:
from pettingzoo.utils.conversions import parallel_to_aec
from supersuit import pettingzoo_env_to_vec_env_v1, concat_vec_envs_v1
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor
from stable_baselines3.common.env_util import make_vec_env

# Create parallel env
env = ParallelMultiAgentGridWorld(grid_size=5, num_agents=2, num_goals=2, seed=42)

# Wrap with vectorized environment
vec_env = concat_vec_envs_v1(env, num_vec_envs=2, num_cpus=1)
vec_env = VecMonitor(vec_env)

# Use PPO with shared policy
model = PPO("MlpPolicy", vec_env, verbose=1)
model.learn(total_timesteps=100_000)

# Save the model
model.save("ppo_multi_agent_gridworld")


Using cuda device


/app/venvs/An/lib/python3.12/site-packages/stable_baselines3/common/vec_env/base_vec_env.py:78: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")


AssertionError: The algorithm only supports (<class 'gymnasium.spaces.box.Box'>, <class 'gymnasium.spaces.discrete.Discrete'>, <class 'gymnasium.spaces.multi_discrete.MultiDiscrete'>, <class 'gymnasium.spaces.multi_binary.MultiBinary'>) as action spaces but <bound method ParallelEnv.action_space of <__main__.ParallelMultiAgentGridWorld object at 0x7b63ee50a600>> was provided